# From RNN states to self-attention
## English sentence reconstruction with three fully connected projections

**Input → contextual representations → word predictions.** This is same-position reconstruction, not translation between languages and not free-length sentence generation.

We first reconstruct four visible words. Then a separate model reconstructs a hidden animal word from the other words. Every step is exposed: data, shapes, Q/K/V, scores, row softmax, weighted values, encoder block, output head, loss, training and evaluation.

**Architecture:** four positions; 450 input/model features; one attention head with 64 query/key/value features; a 450 → 900 → 450 feed-forward network. One head keeps the arithmetic visible; a standard multi-head Transformer extends this operation.

Run all cells in order in local Jupyter or Google Colab. CPU is sufficient. No dataset download or account is needed. The small generated grammar is for inspecting the mechanism, not a language benchmark.

In [ ]:
import importlib.util, subprocess, sys
required = ['torch', 'numpy', 'pandas', 'matplotlib']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
from pathlib import Path
import random, math, time, copy, json, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from IPython.display import display, HTML
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RESULTS = Path('results_english_reconstruction'); RESULTS.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi': 115, 'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False})
display(HTML('<style>.text_cell_render,.jp-MarkdownOutput{font-family:Arial,sans-serif;font-size:18px;line-height:1.85;max-width:1100px}.text_cell_render p{margin:18px 0}.text_cell_render h2{margin-top:40px;color:#17324d}</style>'))
print('PyTorch:', torch.__version__, '| device:', DEVICE)

## 1 · The connection to the RNN lesson
A forward RNN computes $h_i=f(x_i,h_{i-1})$. Its state at position 2 has seen positions 1 and 2, but not positions 3 and 4. A shared output head can already predict a label or word from each RNN state; a decoder is not required simply to attach predictions.

An unmasked self-attention encoder can use all four positions at each position. A bidirectional RNN also provides both-sided context, so this benefit is not exclusive to attention. Attention removes the recurrence across positions within a layer.

The representations have a similar **role**, not identical values or identical computation. A separate sequence decoder is useful when generating another sequence with different length/order. Here the four output positions are already specified.

In [ ]:
def architecture_diagram():
    fig, ax = plt.subplots(figsize=(15, 9)); ax.set(xlim=(0,15), ylim=(0,9)); ax.axis('off')
    def box(x,y,w,h,text,color='#def3f0'):
        ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle='round,pad=.04',fc=color,ec='#087f8c',lw=1.5))
        ax.text(x+w/2,y+h/2,text,ha='center',va='center',fontsize=11)
    def arrow(a,b):ax.annotate('',xy=b,xytext=a,arrowprops=dict(arrowstyle='->',color='#087f8c',lw=1.8))
    box(.2,6.3,2.8,1.2,'X: four positions × 450\nEmbedding + position','#dcecf6')
    for y,name in [(7.6,'Q'),(5.9,'K'),(4.2,'V')]:
        box(3.8,y,3,1.05,f'Linear(450, 64) → {name}\n4 × 64'); arrow((3,6.9),(3.8,y+.525))
    box(7.7,6.5,3,1.1,'QKᵀ / √64\n4 × 4 scores','#fff0c7')
    arrow((6.8,8.1),(7.7,7.3)); arrow((6.8,6.4),(7.7,6.8))
    box(11.6,6.5,3,1.1,'Row softmax → A\n4 × 4 mixing weights','#fff0c7');arrow((10.7,7.05),(11.6,7.05))
    box(9,4.2,4,1.05,'C = A V: 4 × 64\nOne context vector per position')
    arrow((6.8,4.725),(9,4.725));arrow((13.1,6.5),(11,5.25))
    box(5.5,2,8.8,1.15,'Project 64 → 450; add X + LayerNorm\nFFN 450 → 900 → 450; residual + LayerNorm\nH: four contextual vectors, each 450 features','#dcecf6');arrow((11,4.2),(11,3.15))
    box(.2,2,4.3,1.15,'Analogous role to RNN outputs\nH1, H2, H3, H4\nNot recurrent states')
    ax.plot([4.5,5.5],[2.575,2.575],ls='--',color='#b68013')
    box(5.5,.2,8.8,1,'Shared Linear(450, vocabulary) → four word predictions\nCross-entropy with original token IDs','#fff0c7');arrow((10,2),(10,1.2))
    fig.tight_layout();plt.show()
architecture_diagram()

## 2 · Make a small, inspectable English dataset
Each sentence has four words: **adjective · animal · verb · food**. Examples include “small cats eat fish” and “young cows enjoy grass”. The input and clean target have the same positions.

The animal–food associations are deliberately restricted. This creates a learnable context task when the animal is hidden. It is a synthetic grammar, not evidence of general English understanding. The test split holds out exact sentence combinations; vocabulary and grammar are shared across splits.

In [ ]:
adjectives = ['small','young','hungry','happy','quiet','gentle','sleepy','lively',
              'playful','curious','healthy','active','little','calm','friendly','eager']
foods = {'cats':['fish','tuna'], 'dogs':['biscuits','bones'], 'cows':['grass','hay'],
         'birds':['seeds','worms'], 'rabbits':['carrots','lettuce'],
         'horses':['oats','apples'], 'pandas':['bamboo','shoots'], 'monkeys':['bananas','mangoes']}
sentences = [' '.join([adj, animal, verb, food])
             for adj in adjectives for animal, options in foods.items()
             for verb in ['eat','enjoy'] for food in options]
rng = np.random.default_rng(SEED)
order = rng.permutation(len(sentences))
a, b = int(.8*len(order)), int(.9*len(order))
train_text = [sentences[i] for i in order[:a]]
val_text = [sentences[i] for i in order[a:b]]
test_text = [sentences[i] for i in order[b:]]
assert not (set(train_text)&set(val_text) or set(train_text)&set(test_text) or set(val_text)&set(test_text))
# Learn vocabulary from training only; MASK and UNK are special tokens.
vocab = ['[UNK]', '[MASK]'] + sorted(set(' '.join(train_text).split()))
to_id = {word:i for i,word in enumerate(vocab)}
def encode(texts):return torch.tensor([[to_id.get(w,0) for w in s.split()] for s in texts])
def decode(ids):return ' '.join(vocab[int(i)] for i in ids)
train_y, val_y, test_y = map(encode, [train_text,val_text,test_text])
assert not (train_y.eq(0).any() or val_y.eq(0).any() or test_y.eq(0).any())
display(pd.DataFrame({'split':['train','validation','test'], 'sentences':[len(train_y),len(val_y),len(test_y)]}))
display(pd.DataFrame({'example':train_text[:8]}))
fig,ax=plt.subplots(figsize=(8,3));ax.bar(['Train','Validation','Test'],[len(train_y),len(val_y),len(test_y)],color=['#087f8c','#b68013','#7395b0'])
ax.set(ylabel='Four-word sentences',title=f'{len(sentences)} unique sentences; vocabulary size {len(vocab)}');plt.show()

## 3 · Define the two input–target pairs
**Exercise A: visible reconstruction.** All input words are visible; all four original words are targets. This can be solved by copying, so high accuracy does not establish meaningful attention.

**Exercise B: masked reconstruction.** Hide only the animal at position 2. Keep the original sentence as target storage, but calculate loss **only at the hidden position**. The surrounding words are visible evidence, not targets fed into the hidden position. We train a fresh model for this exercise.

All sequences already have exactly four words. No interpolation, padding, start token, end token or causal mask is needed for this same-position task.

In [ ]:
def hide_animal(clean):
    corrupted = clean.clone()
    corrupted[:,1] = to_id['[MASK]']
    return corrupted
train_masked = hide_animal(train_y)
val_masked = hide_animal(val_y)
test_masked = hide_animal(test_y)
display(pd.DataFrame({'visible_input':train_text[:5],
                      'masked_input':[decode(x) for x in train_masked[:5]],
                      'clean_target':train_text[:5],
                      'scored_masked_target':[vocab[int(x[1])] for x in train_y[:5]]}))
print('Token ID tensor:',tuple(train_y.shape),'= sentences × positions')

## 4 · Embedding plus position: 4 × 450
The embedding layer selects a learned 450-feature vector for each token ID. A learned position embedding adds the position identity. The sum remains 450 features. Position vectors are learned here; the earlier full translation notebook used sinusoidal positions. Both are valid choices.

The three projections share weights across positions, but have different weights from one another. This is three linear layers, not three separately trained models.

In [ ]:
D_MODEL, D_HEAD, D_FF, LENGTH = 450, 64, 900, 4
class InputEmbedding(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.words = nn.Embedding(vocab_size,D_MODEL)
        self.positions = nn.Embedding(LENGTH,D_MODEL)
    def forward(self, ids):
        positions = torch.arange(ids.shape[1],device=ids.device)
        return self.words(ids) + self.positions(positions)[None,:,:]
example_embedding = InputEmbedding(len(vocab))
example_x = example_embedding(train_y[:1])
print('One sentence:',tuple(example_x.shape),'= batch × positions × features')

## 5 · Three linear layers, pairwise scores, then mixing
Using row-vector notation, $Q=XW_Q+b_Q$, and similarly for $K,V$. PyTorch stores each Linear weight transposed relative to this notation.

$E=QK^T/\sqrt{64}$ has shape $4\times4$. Entry $(i,j)$ is the comparison of query **position i** with key **position j**, by construction. Row softmax produces nonnegative weights summing to one.

$C=AV$ produces four context vectors with 64 features each. The weights are normalized mixing coefficients, not calibrated probabilities of grammatical dependence. Each row belongs to its query position; this indexing is guaranteed. Whether the learned mixture is useful depends on training.

Scaling by $\sqrt{64}$ controls score magnitudes. The value projection decides which features to share. Queries and keys decide how to match positions. There are no manual target matrices for any of them.

In [ ]:
class SingleHeadSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.query = nn.Linear(D_MODEL,D_HEAD)
        self.key = nn.Linear(D_MODEL,D_HEAD)
        self.value = nn.Linear(D_MODEL,D_HEAD)
    def forward(self, x):
        q = self.query(x)                         # B × 4 × 64
        k = self.key(x)                           # B × 4 × 64
        v = self.value(x)                         # B × 4 × 64
        scores = q @ k.transpose(-2,-1) / math.sqrt(D_HEAD)
        weights = torch.softmax(scores,dim=-1)    # B × 4 × 4
        context = weights @ v                    # B × 4 × 64
        return context, {'Q':q,'K':k,'V':v,'scores':scores,'A':weights,'C':context}
attention_example = SingleHeadSelfAttention()
example_c, example_trace = attention_example(example_x)
for name,tensor in example_trace.items():print(name,tuple(tensor.shape))
assert torch.allclose(example_trace['A'].sum(-1),torch.ones(1,4),atol=1e-6)
manual_c2 = sum(example_trace['A'][0,1,j]*example_trace['V'][0,j] for j in range(4))
assert torch.allclose(manual_c2,example_c[0,1],atol=1e-6)
print('Verified: row sums = 1; manual weighted sum equals context at position 2.')

## 6 · Complete the encoder, then attach the word-prediction head
Project the context vectors from 64 back to 450 features so they can be added to the original input. The first residual/LayerNorm yields U. Apply a position-wise FFN and another residual/LayerNorm to obtain H.

$$U=\mathrm{LayerNorm}(X+CW_O+b_O)$$
$$H=\mathrm{LayerNorm}(U+\mathrm{FFN}(U))$$

This explicit post-normalization block omits dropout for a minimal deterministic exercise. One head is used, rather than the multiple heads of a typical Transformer.

H has four contextual vectors, each 450-dimensional. It is not the target and it is not an RNN recurrence. A shared Linear(450, vocabulary size) produces one word distribution per position. No separate Transformer decoder is required; this is an encoder with a minimal reconstruction head, not a general sequence generator.

In [ ]:
class ReconstructionEncoder(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = InputEmbedding(vocab_size)
        self.attention = SingleHeadSelfAttention()
        self.output_projection = nn.Linear(D_HEAD,D_MODEL)
        self.norm1 = nn.LayerNorm(D_MODEL)
        self.ffn = nn.Sequential(nn.Linear(D_MODEL,D_FF),nn.ReLU(),nn.Linear(D_FF,D_MODEL))
        self.norm2 = nn.LayerNorm(D_MODEL)
        self.word_head = nn.Linear(D_MODEL,vocab_size)
    def forward(self, ids):
        x = self.embedding(ids)
        context, trace = self.attention(x)
        u = self.norm1(x + self.output_projection(context))
        h = self.norm2(u + self.ffn(u))
        logits = self.word_head(h)
        return logits, {**trace,'X':x,'U':u,'H':h}
shape_model = ReconstructionEncoder(len(vocab))
logits,trace = shape_model(train_y[:2])
display(pd.DataFrame([{'tensor':k,'shape':str(tuple(v.shape))} for k,v in trace.items()]))
print('Logits:',tuple(logits.shape),'= batch × positions × vocabulary')
print('Trainable parameters:',sum(p.numel() for p in shape_model.parameters()))
print(shape_model)

## 7 · What is the training target? Follow the gradient
For visible reconstruction, cross-entropy uses all four original token IDs. For masked reconstruction, use only the original animal ID at position 2. **Do not apply softmax before PyTorch CrossEntropyLoss.** It expects logits.

Loss flows through the word head, H, the attention output and its projections. For Q and K the path includes the attention softmax; for V it includes the weighted sum. These internal quantities have no separate labels. Residual paths also carry gradients.

A batch update changes shared parameters jointly. Parallel forward computation across positions does not mean four independent models or four separate targets for attention.

In [ ]:
criterion = nn.CrossEntropyLoss()
def task_loss(logits, targets, masked=False):
    if masked:
        return criterion(logits[:,1,:],targets[:,1])
    return criterion(logits.reshape(-1,logits.shape[-1]),targets.reshape(-1))
probe = ReconstructionEncoder(len(vocab))
probe_logits,_ = probe(train_masked[:8])
probe_loss = task_loss(probe_logits,train_y[:8],masked=True)
probe_loss.backward()
gradients = {name:float(layer.weight.grad.norm()) for name,layer in
             [('query',probe.attention.query),('key',probe.attention.key),('value',probe.attention.value)]}
print('Untrained masked loss:',float(probe_loss.detach()))
print('Gradient norms:',gradients)
assert all(value>0 for value in gradients.values())
print('All three projection layers receive a training signal from word loss.')

## 8 · Shared training utilities
The two experiments below instantiate and train their own models. This helper only handles batching, validation and best-checkpoint restoration. Validation selects the checkpoint; test sentences remain unused until final evaluation. Batch size 64 means one optimizer update after the loss for that batch.

In [ ]:
def train_model(model, inputs, targets, val_inputs, val_targets, masked, epochs):
    loader = DataLoader(TensorDataset(inputs,targets),batch_size=64,shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(),lr=0.0005)
    history=[]; best=float('inf'); best_state=None; start=time.perf_counter()
    for epoch in range(1,epochs+1):
        model.train(); total=0
        for batch_x,batch_y in loader:
            batch_x,batch_y=batch_x.to(DEVICE),batch_y.to(DEVICE)
            optimizer.zero_grad()
            logits,_=model(batch_x)
            loss=task_loss(logits,batch_y,masked)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
            total+=loss.item()*len(batch_x)
        model.eval()
        with torch.no_grad():
            vl,_=model(val_inputs.to(DEVICE))
            val_loss=task_loss(vl,val_targets.to(DEVICE),masked).item()
        history.append({'epoch':epoch,'train_loss':total/len(inputs),'val_loss':val_loss})
        if val_loss<best:best=val_loss;best_state=copy.deepcopy(model.state_dict());best_epoch=epoch
        if epoch==1 or epoch%10==0:print(f'Epoch {epoch:3d} | train {total/len(inputs):.4f} | validation {val_loss:.4f}')
    model.load_state_dict(best_state)
    elapsed=time.perf_counter()-start
    print(f'Restored epoch {best_epoch}; training time {elapsed:.1f} s')
    return pd.DataFrame(history),elapsed,best_epoch

def show_history(history,title):
    fig,ax=plt.subplots(figsize=(8,3.5));ax.plot(history.epoch,history.train_loss,label='Training');ax.plot(history.epoch,history.val_loss,label='Validation')
    ax.set(xlabel='Epoch',ylabel='Cross-entropy',title=title);ax.legend();plt.show()

## 9 · Experiment A: reconstruct the visible sentence
This fresh model sees every target word in its input. Good reconstruction is expected and is useful for checking the pipeline. It is not a demonstration that cross-word reasoning was necessary.

In [ ]:
torch.manual_seed(SEED)
copy_model = ReconstructionEncoder(len(vocab)).to(DEVICE)
copy_history,copy_seconds,copy_epoch = train_model(copy_model,train_y,train_y,val_y,val_y,masked=False,epochs=20)
show_history(copy_history,'A · Visible sentence reconstruction')

## 10 · Experiment B: recover a hidden animal from context
Start a fresh model. Input: “small [MASK] eat fish”. Target at position 2: “cats”. The visible food is deliberately informative in this controlled grammar.

Only the masked position contributes to loss. Predictions at the other positions are not trained by this objective. To display a reconstructed sentence, preserve visible tokens and fill the masked position with its prediction.

In [ ]:
torch.manual_seed(SEED+1)
masked_model = ReconstructionEncoder(len(vocab)).to(DEVICE)
masked_history,masked_seconds,masked_epoch = train_model(masked_model,train_masked,train_y,val_masked,val_y,masked=True,epochs=50)
show_history(masked_history,'B · Masked animal reconstruction')

## 11 · Evaluate the held-out sentences
Report visible-reconstruction token accuracy and whole-sentence accuracy. For the masked task, report only masked-position accuracy. A fully filled sentence is correct exactly when its one hidden token is correct; visible words are copied from the input.

These results measure held-out combinations within a tiny generated grammar. They do not measure unrestricted English comprehension. Accuracy alone cannot validate a linguistic interpretation of an attention map.

In [ ]:
copy_model.eval();masked_model.eval()
with torch.no_grad():
    copy_logits,_=copy_model(test_y.to(DEVICE))
    mask_logits,_=masked_model(test_masked.to(DEVICE))
copy_pred=copy_logits.argmax(-1).cpu(); mask_pred=mask_logits.argmax(-1).cpu()
filled=test_masked.clone();filled[:,1]=mask_pred[:,1]
metrics={'sentences':len(sentences),'train':len(train_y),'validation':len(val_y),'test':len(test_y),
         'vocabulary':len(vocab),'parameters':sum(p.numel() for p in masked_model.parameters()),
         'copy_token_accuracy':float((copy_pred==test_y).float().mean()),
         'copy_sentence_accuracy':float((copy_pred==test_y).all(1).float().mean()),
         'masked_accuracy':float((mask_pred[:,1]==test_y[:,1]).float().mean()),
         'copy_seconds':copy_seconds,'masked_seconds':masked_seconds,
         'copy_best_epoch':copy_epoch,'masked_best_epoch':masked_epoch}
display(pd.DataFrame({'metric':['Copy token accuracy','Copy sentence accuracy','Masked-position accuracy'],
                      'percent':[100*metrics[k] for k in ['copy_token_accuracy','copy_sentence_accuracy','masked_accuracy']]}))
examples=pd.DataFrame({'input':[decode(x) for x in test_masked[:12]],'target':test_text[:12],'filled_output':[decode(x) for x in filled[:12]]})
display(examples)
(RESULTS/'metrics.json').write_text(json.dumps(metrics,indent=2))
copy_history.to_csv(RESULTS/'copy_history.csv',index=False);masked_history.to_csv(RESULTS/'masked_history.csv',index=False)
examples.to_csv(RESULTS/'examples.csv',index=False)
torch.save({'state_dict':masked_model.state_dict(),'vocab':vocab,'dimensions':[D_MODEL,D_HEAD,D_FF]},RESULTS/'masked_model.pt')

## 12 · Inspect the trained matrices, row by row
Choose one held-out example. Print Q, K, V and all four attention rows. Row 2 belongs to the hidden position because it comes from that position’s query. Its weighted sum yields context vector C2.

The matrix is not supervised with an alignment target. Large entries are actual mixing weights, not proof that the corresponding word is the unique cause of the final prediction. Values and residual paths matter too.

In [ ]:
sample = test_masked[:1].to(DEVICE)
with torch.no_grad():sample_logits,learned=masked_model(sample)
labels=decode(sample[0].cpu()).split()
for name in ['Q','K','V']:
    print(name,'first 6 of 64 features')
    display(pd.DataFrame(learned[name][0,:,:6].cpu().numpy(),index=labels).round(3))
attention=learned['A'][0].cpu().numpy()
display(pd.DataFrame(attention,index=[f'query {i+1}: {w}' for i,w in enumerate(labels)],columns=[f'key {i+1}: {w}' for i,w in enumerate(labels)]).round(4))
fig,ax=plt.subplots(figsize=(7,5));im=ax.imshow(attention,vmin=0,vmax=1,cmap='YlGnBu')
ax.set_xticks(range(4),labels,rotation=20);ax.set_yticks(range(4),labels)
ax.set(xlabel='Key / value position',ylabel='Query / output position',title='Learned attention · masked reconstruction')
for i in range(4):
    for j in range(4):ax.text(j,i,f'{attention[i,j]:.3f}',ha='center',va='center',color='white' if attention[i,j]>.5 else 'black')
fig.colorbar(im,ax=ax,label='Mixing weight');fig.tight_layout();plt.show()
manual=sum(learned['A'][0,1,j]*learned['V'][0,j] for j in range(4))
assert torch.allclose(manual,learned['C'][0,1],atol=1e-5)
print('C2 first 8 features:',manual[:8].cpu().numpy().round(3))
print('Prediction:',vocab[int(sample_logits[0,1].argmax())], '| target:',vocab[int(test_y[0,1])])
(RESULTS/'attention_example.json').write_text(json.dumps({'input':labels,'target':test_text[0],'weights':attention.tolist(),'prediction':vocab[int(sample_logits[0,1].argmax())]}))

## 13 · Verify information flow, not just a picture
We can check row normalization, the exact context calculation and gradients. We can also remove the attention contribution at inference. In this one-block model, a hidden position then sees only its MASK embedding and position through the residual path: other words have no route to it.

This ablation demonstrates the role of cross-position mixing in this trained model. It does not prove that every individual attention weight is a faithful causal explanation. The ablation changes the distribution of internal activations.

In [ ]:
with torch.no_grad():
    x=masked_model.embedding(test_masked.to(DEVICE))
    zero_context=torch.zeros(len(x),4,D_HEAD,device=DEVICE)
    u=masked_model.norm1(x+masked_model.output_projection(zero_context))
    h=masked_model.norm2(u+masked_model.ffn(u))
    no_mix=masked_model.word_head(h).argmax(-1).cpu()
no_mix_accuracy=float((no_mix[:,1]==test_y[:,1]).float().mean())
print(f'Normal masked accuracy: {100*metrics["masked_accuracy"]:.2f}%')
print(f'Attention contribution removed: {100*no_mix_accuracy:.2f}%')
metrics['masked_no_attention_accuracy']=no_mix_accuracy
(RESULTS/'metrics.json').write_text(json.dumps(metrics,indent=2))
fig,ax=plt.subplots(figsize=(7,3.5));ax.bar(['Normal encoder','Zero attention context'],[100*metrics['masked_accuracy'],100*no_mix_accuracy],color=['#087f8c','#b68013'])
ax.set(ylim=(0,105),ylabel='Masked-position accuracy (%)',title='Does cross-position mixing help this model?');plt.show()

## 14 · Try another sentence
Use the controlled vocabulary and four-position grammar. This model predicts only the animal at position 2. It is not a general chatbot, translator or sentence continuation model.

In [ ]:
custom = 'happy [MASK] enjoy bamboo'
words=custom.split()
assert len(words)==4 and words[1]=='[MASK]'
unknown=[word for word in words if word not in to_id]
assert not unknown, f'Unknown vocabulary: {unknown}'
with torch.no_grad():custom_logits,_=masked_model(encode([custom]).to(DEVICE))
result=words.copy();result[1]=vocab[int(custom_logits[0,1].argmax())]
print('Input: ',custom)
print('Filled:',' '.join(result))

## 15 · The bridge to translation
We have learned how an encoder creates a contextual vector at every position and how a simple head can use those vectors. This is an encoder with a reconstruction head, not an encoder and sequence decoder merged into one block.

For translation, output length and order can differ. A Transformer decoder introduces target-prefix representations, causal self-attention and cross-attention to the encoder memory. That is the next exercise.

**References:** [Attention Is All You Need](https://arxiv.org/abs/1706.03762), [PyTorch Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html), [PyTorch CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html). Numerical dimensions and the controlled grammar here are illustrative design choices, not the original paper configuration.